# Hestia P4 — larger-category correction

**Actual CUDA inference, not a hosted text API.** Run on a fresh Colab GPU
(T4 with sufficient free VRAM or a larger GPU). This tests Falcon3-7B plus
Qwen2.5-14B, Phi-3-medium-4k, and Qwen3-14B: four models × two fixed inputs ×
zero/one/few/many-shot = **32 generations**.

Weights are NF4 4-bit; compute and non-quantized modules are FP32 to avoid the
FP16 numerical issue observed in P3's Qwen work. Attention is eager, generation
is deterministic with 220 new tokens, Qwen3 thinking is disabled. These results
are labeled quantized and must not be passed off as native FP16 measurements.
The runner checks finite logits, records revisions/configuration and timings,
and checkpoints every response. Quality scores stay blank until outputs are read.

No Hugging Face token, paid API key, Drive mounting, or paid Colab purchase is
required for these public models. Use only an available GPU session; do not buy
compute units as part of running this notebook. Downloads can be large. Model
weights are deleted from this notebook's dedicated cache after each successful
model; results are retained. Original small-model results remain separate.

P1 owns the final matrix. These runs supply one-large / three-above-10B candidates
under the original ranges. Main PR #31 now uses different balanced ranges;
these tests are separate evidence until P1/the team chooses the final selection.
Do not blindly add them to the existing 15-model notebook.


In [ ]:
# Environment setup; first generation below downloads the actual model weights.
%pip -q install transformers==4.57.6 huggingface-hub==0.36.2 accelerate==1.15.0 bitsandbytes==0.50.2 'sentencepiece>=0.2,<0.3' 'protobuf>=6,<7' 'psutil>=7,<8'


In [ ]:
import os, sys, json, shutil, subprocess, zipfile
from pathlib import Path
os.environ['HF_HOME']='/content/hestia_p4_cache'
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
os.environ['HF_HUB_DISABLE_XET']='1'
import torch
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > GPU first. CPU is not sufficient.'
print('GPU:', torch.cuda.get_device_name(0))
print('Free/total VRAM GiB:', tuple(round(x/2**30,2) for x in torch.cuda.mem_get_info()))
WORK=Path('/content/hestia_p4_large')
WORK.mkdir(exist_ok=True)
OUTPUT=WORK/'a6_results/person4_large'
OUTPUT.mkdir(parents=True,exist_ok=True)
RUNNER=WORK/'person4_large_benchmark.py'
RUNNER.write_text('"""P4 category correction: four real local-to-GPU Transformers benchmarks.\n\nUses the unchanged P1/P2 prompt/schema contract; NF4 weight quantization with\nFP32 compute and non-quantized modules. No paid provider, no fine-tuning.\nEach model runs in its own process when launched by the accompanying notebook.\n"""\nimport argparse\nimport csv\nimport gc\nimport hashlib\nimport importlib.util\nimport inspect\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport time\nfrom datetime import datetime, timezone\n\nMODELS = [\n    "tiiuae/Falcon3-7B-Instruct",\n    "Qwen/Qwen2.5-14B-Instruct",\n    "microsoft/Phi-3-medium-4k-instruct",\n    "Qwen/Qwen3-14B",\n]\nCONTRACT_SHA256 = "8403f9e97778af00d8b389cb8dce2a9d8fa94747cb737820883c4623c83cd459"\nCONTRACT_COMMIT = "14a02aeb4eb71d1ddf7dfb66d1e868b77213eb28"\n\n\ndef shared_contract(folder):\n    """Load pinned team-owned Python helpers, not model-generated code."""\n    folder.mkdir(parents=True, exist_ok=True)\n    source = folder / "team_contract.py"\n    if not source.exists():\n        import requests\n        url = ("https://raw.githubusercontent.com/HestiaaAI/12_HestiaaAI/"\n               f"{CONTRACT_COMMIT}/docs/a6-person-2/run_shivani_contract_benchmark.py")\n        response = requests.get(url, timeout=45)\n        response.raise_for_status()\n        source.write_bytes(response.content)\n    if hashlib.sha256(source.read_bytes()).hexdigest() != CONTRACT_SHA256:\n        raise RuntimeError("Shared contract SHA-256 mismatch; refusing to run changed prompts")\n    spec = importlib.util.spec_from_file_location("hestia_shared_contract", source)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n\n\ndef persist(rows, fields, csv_path):\n    temporary = csv_path.with_suffix(".tmp")\n    with temporary.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fields, lineterminator="\\n")\n        writer.writeheader()\n        writer.writerows(rows)\n    temporary.replace(csv_path)\n\n\ndef run(model_id, output):\n    import torch\n    import transformers\n    import bitsandbytes\n    import psutil\n    from huggingface_hub import HfApi\n    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n\n    output.mkdir(parents=True, exist_ok=True)\n    team = shared_contract(output / "support")\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA GPU required. This is not a completed benchmark on CPU.")\n    gpu = torch.cuda.get_device_properties(0)\n    free, total = torch.cuda.mem_get_info()\n    if free < 13.0 * 2**30 and "14B" in model_id:\n        raise RuntimeError(f"Need at least 13 GiB free VRAM for this 14B configuration; available {free/2**30:.2f} GiB")\n    torch.manual_seed(0)\n    torch.cuda.manual_seed_all(0)\n    torch.set_num_threads(2)\n    csv_path = output / "person4_large_results.csv"\n    rows = []\n    if csv_path.exists():\n        with csv_path.open(encoding="utf-8", newline="") as handle:\n            rows = list(csv.DictReader(handle))\n    done = {(row["model_id"], row["input_id"], row["shot_mode"]) for row in rows}\n    expected = {(model_id, key, shot) for key in team.TEST_INPUTS for shot in team.SHOT_MODES}\n    if expected <= done:\n        print("All eight actual generations already checkpointed for", model_id, flush=True)\n        return\n    sha = HfApi().model_info(model_id).sha\n    metadata_path = output / "person4_large_metadata.json"\n    metadata = json.loads(metadata_path.read_text(encoding="utf-8")) if metadata_path.exists() else {"models": {}}\n    previous = metadata["models"].get(model_id, {})\n    sha = previous.get("revision", sha)\n    metadata.update({"created_at_utc": metadata.get("created_at_utc", datetime.now(timezone.utc).isoformat()),\n        "python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,\n        "bitsandbytes": bitsandbytes.__version__, "device": "cuda:0", "gpu": gpu.name,\n        "vram_gib": round(total/2**30, 3), "host_ram_gib": round(psutil.virtual_memory().total/2**30, 3),\n        "quantization": "4-bit NF4 weights with double quantization",\n        "compute_dtype": "float32", "non_quantized_dtype": "float32", "attention": "eager",\n        "max_new_tokens": 220, "do_sample": False, "seed": 0, "contract_sha256": CONTRACT_SHA256,\n        "timing_note": "Load includes initial downloads when uncached; generation synchronized on CUDA. Cross-hardware/precision speed comparisons are not controlled.",\n        "review_attribution": "Unscored until raw-output content review; no generated or heuristic scores.",\n        "configurations_comparable_to_native": False})\n    metadata["models"][model_id] = {"revision": sha, "status": "loading", "qwen3_thinking": False if "Qwen3" in model_id else None}\n    metadata_path.write_text(json.dumps(metadata, indent=2) + "\\n", encoding="utf-8")\n    print("Loading", model_id, "revision", sha, "on", gpu.name, flush=True)\n    start = time.perf_counter()\n    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=sha, trust_remote_code=False)\n    model = AutoModelForCausalLM.from_pretrained(model_id, revision=sha, trust_remote_code=False,\n        dtype=torch.float32, device_map={"": 0}, low_cpu_mem_usage=True,\n        attn_implementation="eager", quantization_config=BitsAndBytesConfig(\n            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,\n            bnb_4bit_compute_dtype=torch.float32))\n    model.eval()\n    torch.cuda.synchronize()\n    load_seconds = time.perf_counter() - start\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token_id = tokenizer.eos_token_id\n    metadata["models"][model_id].update(status="generating", load_seconds=round(load_seconds,3),\n                                         footprint_gib=round(model.get_memory_footprint()/2**30,3))\n    metadata_path.write_text(json.dumps(metadata, indent=2) + "\\n", encoding="utf-8")\n    for input_id, text in team.TEST_INPUTS.items():\n        for shot in team.SHOT_MODES:\n            if (model_id, input_id, shot) in done:\n                continue\n            messages = [{"role":"system", "content":team.SYSTEM_INSTRUCTION},\n                        {"role":"user", "content":team.build_user_prompt(text,shot)}]\n            options = {"enable_thinking": False} if "Qwen3" in model_id else {}\n            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **options)\n            encoded = tokenizer(prompt, return_tensors="pt").to("cuda:0")\n            forward_fields = inspect.signature(model.forward).parameters\n            last_logit = ({"logits_to_keep": 1} if "logits_to_keep" in forward_fields else\n                          {"num_logits_to_keep": 1} if "num_logits_to_keep" in forward_fields else {})\n            with torch.inference_mode():\n                probe = model(**encoded, use_cache=False, **last_logit)\n                finite = bool(torch.isfinite(probe.logits[:, -1, :]).all())\n                del probe\n            if not finite:\n                raise RuntimeError(f"Non-finite pre-generation logits: {model_id} {input_id}/{shot}; no output fabricated")\n            torch.cuda.reset_peak_memory_stats()\n            torch.cuda.synchronize()\n            start = time.perf_counter()\n            with torch.inference_mode():\n                generated = model.generate(**encoded, max_new_tokens=220, do_sample=False,\n                    pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)\n            torch.cuda.synchronize()\n            seconds = time.perf_counter() - start\n            tokens = generated[0, encoded["input_ids"].shape[-1]:]\n            raw = tokenizer.decode(tokens, skip_special_tokens=True).strip()\n            try:\n                errors = team.validate_payload(team.extract_json(raw))\n            except (ValueError, TypeError, json.JSONDecodeError) as exc:\n                errors = [str(exc)]\n            rows.append({"model_id":model_id, "input_id":input_id, "shot_mode":shot,\n                "load_seconds":round(load_seconds,3), "generation_seconds":round(seconds,3),\n                "output_tokens":int(tokens.numel()), "schema_valid":not errors,\n                "validation_errors":" | ".join(errors), "raw_output":raw,\n                "quality_score_0_to_10":"", "review_notes":"NF4 weights; FP32 compute/non-quantized modules; eager CUDA; pending raw-output review."})\n            persist(rows,team.CSV_FIELDS,csv_path)\n            metadata["models"][model_id].update(last_completed=f"{input_id}/{shot}",\n                peak_allocated_gib=round(torch.cuda.max_memory_allocated()/2**30,3))\n            metadata_path.write_text(json.dumps(metadata,indent=2)+"\\n",encoding="utf-8")\n            print(f"CHECKPOINT {model_id} {input_id}/{shot}: {seconds:.3f}s; schema={not errors}; tokens={tokens.numel()}",flush=True)\n            del generated, tokens, encoded\n            gc.collect()\n    metadata["models"][model_id].update(status="complete", completed_generations=8)\n    metadata_path.write_text(json.dumps(metadata,indent=2)+"\\n",encoding="utf-8")\n    print("MODEL COMPLETE:",model_id,flush=True)\n\n\nif __name__ == "__main__":\n    parser=argparse.ArgumentParser()\n    parser.add_argument("--model",choices=MODELS,required=True)\n    parser.add_argument("--output",type=Path,default=Path("a6_results/person4_large"))\n    args=parser.parse_args()\n    run(args.model,args.output)\n',encoding='utf-8')

def package_results():
    bundle=Path('/content/hestia_p4_large_results.zip')
    with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT.rglob('*'):
            if p.is_file() and '__pycache__' not in p.parts:
                z.write(p,p.relative_to(WORK))
        z.write(RUNNER,'person4_large_benchmark.py')
    return str(bundle)

def run_one(model):
    # Separate process releases all model/allocator memory between models.
    with subprocess.Popen([sys.executable,'-u',str(RUNNER),'--model',model,'--output',str(OUTPUT)],
                          stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        for line in process.stdout:
            print(line,end='',flush=True)
        code=process.wait()
    print('Checkpoint bundle:',package_results(),flush=True)
    if code:
        raise RuntimeError(f'{model} stopped (exit {code}); retain partial results and inspect the error. Do not count it as completed.')
    # Only our newly downloaded, reproducible weight cache is removed.
    cache=Path('/content/hestia_p4_cache').resolve()
    folder=cache/'hub'/('models--'+model.replace('/','--'))
    if folder.resolve().is_relative_to(cache) and folder.exists():
        shutil.rmtree(folder)
    print('Completed model; result evidence preserved.',flush=True)


## tiiuae/Falcon3-7B-Instruct

First use downloads weights automatically through Transformers. Eight actual generations are checkpointed.

In [ ]:
run_one('tiiuae/Falcon3-7B-Instruct')

## Qwen/Qwen2.5-14B-Instruct

First use downloads weights automatically through Transformers. Eight actual generations are checkpointed.

In [ ]:
run_one('Qwen/Qwen2.5-14B-Instruct')

## microsoft/Phi-3-medium-4k-instruct

First use downloads weights automatically through Transformers. Eight actual generations are checkpointed.

In [ ]:
run_one('microsoft/Phi-3-medium-4k-instruct')

## Qwen/Qwen3-14B

First use downloads weights automatically through Transformers. Eight actual generations are checkpointed.

In [ ]:
run_one('Qwen/Qwen3-14B')

## Export the actual evidence

Run this cell even after a model failure to preserve partial results. A partial
CSV is not a completed four-model benchmark. Do not invent quality scores.
Return the ZIP to the P4 checkout for raw-output review, metadata validation,
and P1's final model-matrix handoff. The checksum/base64 block is a backup transfer
of the same small evidence ZIP; it contains no weights.

In [ ]:
import csv, base64, hashlib
path=OUTPUT/'person4_large_results.csv'
if path.exists():
    with path.open(encoding='utf-8',newline='') as f:
        rows=list(csv.DictReader(f))
    counts={model:sum(r['model_id']==model for r in rows) for model in [
        'tiiuae/Falcon3-7B-Instruct','Qwen/Qwen2.5-14B-Instruct',
        'microsoft/Phi-3-medium-4k-instruct','Qwen/Qwen3-14B']}
    print('Actual completed generations:',len(rows),counts)
    print('All four complete:',len(rows)==32 and all(n==8 for n in counts.values()))
else:
    print('No actual generations completed yet.')
bundle=Path(package_results())
print('P4_ZIP_SHA256',hashlib.sha256(bundle.read_bytes()).hexdigest())
print('P4_ZIP_BASE64_BEGIN')
print(base64.b64encode(bundle.read_bytes()).decode('ascii'))
print('P4_ZIP_BASE64_END')
from google.colab import files
files.download(str(bundle))
